# Scania APS: data cleaning without losing the failure cases

**An operational data-quality project using pandas.**

This notebook turns the original Scania APS training and test CSVs into traceable,
validated datasets. It investigates missing measurements, duplicate-looking records,
feature consistency and the consequences of common cleaning shortcuts.

**Key findings from the executed analysis**

- Complete-case deletion would retain **591 of 60,000 training records** and only
  **76 of 1,000 APS cases**. Missing data needs a policy, rather than blanket row deletion.
- APS training cases have **17.15%** of measurements missing on average, compared
  with **8.18%** for other-component failure cases. Dropping incomplete records changes
  the composition of the sample.
- Eight features are more than 50% missing in training; one feature has a constant
  observed value. These are review flags, not permission to discard measurements silently.
- The conservative cleaned datasets retain **all 76,000 source records**. A separate
  median-filled preparation example learns its values from training only and records
  where measurements were missing.

No predictive model is fitted. These are findings about this expert-selected dataset,
not estimates of fleet failure rates or claims of maintenance savings.

## 1. Operational context and scope

Scania's air pressure system (APS) supplies functions including braking and gear changes.
The source's `pos` label identifies failure of a specified APS component; `neg` identifies
failure of other components. **Negative does not mean a healthy truck.**

The original challenge assigns a cost of 10 to an unnecessary check and 500 to a missed
APS case: a **50:1 ratio in challenge cost units**, not a quoted monetary saving. This makes
the retention of APS observations an important cleaning consideration. The notebook does
not evaluate a classifier or claim a reduction in that cost.

**Grain and limits:** one supplied observation per CSV row. The files contain anonymised
numerical counters and histogram bins, with no usable truck ID, timestamp, maintenance
history or feature units. A source row number provides lineage only; it is not a truck ID.
Repeated trucks and real-world train/test independence therefore cannot be established.

Source: [UCI APS Failure at Scania Trucks](https://archive.ics.uci.edu/dataset/421/aps+failure+at+scania+trucks),
[DOI: 10.24432/C51S51](https://doi.org/10.24432/C51S51). The original description is downloaded
alongside the CSVs. UCI currently displays CC BY 4.0, while the original files carry a Scania
copyright and GPLv3-or-later notice. Both source notices are retained locally; the data is
downloaded from UCI rather than redistributed in this repository.

## 2. Setup and reproducible source files

Run the notebook from top to bottom, with the kernel started in the repository or its
`notebooks` folder. **Pandas is the only data-analysis library.** The other imports are
Python's standard library for paths, downloading, ZIP files, checksums and JSON.

The first run downloads the official archive. Subsequent runs reuse it. SHA-256 checks
pin the exact source version used here, and the metadata header is located before parsing
the CSV. If UCI changes the archive, the notebook stops for review rather than accepting a
different input silently.

In [1]:
from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import ZipFile
from hashlib import sha256
import json
import shutil
import pandas as pd

pd.set_option("display.max_columns", 9)
pd.set_option("display.max_rows", 15)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks").is_dir() and (p / "requirements.txt").exists()), None)
assert ROOT is not None, "Start Jupyter from this repository or its notebooks folder."
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
for folder in (RAW, PROCESSED, REPORTS):
    folder.mkdir(parents=True, exist_ok=True)

SOURCE_URL = "https://archive.ics.uci.edu/static/public/421/aps+failure+at+scania+trucks.zip"
ARCHIVE_SHA256 = "5504d0402f54faaf97ac0ca085a621645763f5cfea2eb29c592b057d43d4db89"
FILES = {"train": "aps_failure_training_set.csv", "test": "aps_failure_test_set.csv"}
EXPECTED_SHA256 = {
    "aps_failure_training_set.csv": "bb484302e3a3a1c8ef5e1f0129c4dc7cbd58f350867f95b575461ca21ab6b9da",
    "aps_failure_test_set.csv": "2cdf6f7661c7b4c63333c93cdec36a3a82350176b604a2312cf82799fb2712f3",
    "aps_failure_description.txt": "57b1eb3c551a678ffc1c8112e5b34c566dd16457642563f58aeb3ca65beeb820",
}

def file_hash(path):
    with path.open("rb") as handle:
        return sha256(handle.read()).hexdigest()

print(f"Analysis dependency: pandas {pd.__version__}")

Analysis dependency: pandas 2.3.3


In [2]:
archive = RAW / "aps_failure.zip"
if not archive.exists():
    temporary = archive.with_suffix(".download")
    request = Request(SOURCE_URL, headers={"User-Agent": "Scania-APS-data-project/1.0"})
    with urlopen(request, timeout=60) as response, temporary.open("wb") as target:
        shutil.copyfileobj(response, target)
    assert file_hash(temporary) == ARCHIVE_SHA256, "Source archive changed; review before continuing."
    temporary.replace(archive)
assert file_hash(archive) == ARCHIVE_SHA256, "Cached archive checksum mismatch."

with ZipFile(archive) as bundle:
    for filename, expected_hash in EXPECTED_SHA256.items():
        destination = RAW / filename
        if not destination.exists():
            # Only the three expected filenames are written; no arbitrary archive paths.
            destination.write_bytes(bundle.read(filename))
        assert file_hash(destination) == expected_hash, f"Source file changed: {filename}"

source_manifest = pd.DataFrame([
    {"file": name, "bytes": (RAW / name).stat().st_size, "sha256": file_hash(RAW / name)}
    for name in EXPECTED_SHA256
])
source_manifest[["file", "bytes"]]

,file,bytes
0,aps_failure_training_set.csv,44669194
1,aps_failure_test_set.csv,11943558
2,aps_failure_description.txt,4874


## 3. Read the original CSVs and audit the tokens

The source has a 20-line notice before the header. Reading it directly as a normal CSV
would misinterpret that notice. The loader locates the `class,` header, rejects duplicate
column names and loads strings first so unexpected tokens cannot disappear during numeric
conversion. The only accepted missing marker is the documented literal `na`.

Whitespace around labels and measurements is stripped. Feature names are preserved exactly
(including `am_0` and `ec_00`); unusual spelling alone does not justify changing an identifier.

In [3]:
def read_source(filename):
    path = RAW / filename
    with path.open(encoding="utf-8") as handle:
        header_line = next(i for i, line in enumerate(handle) if line.startswith("class,"))
    with path.open(encoding="utf-8") as handle:
        header = handle.read().splitlines()[header_line].split(",")
    assert len(header) == len(set(header)), f"Duplicate source columns: {filename}"
    frame = pd.read_csv(path, skiprows=header_line, dtype="string", keep_default_na=False)
    assert frame.columns.tolist() == header
    frame.index = pd.RangeIndex(1, len(frame) + 1, name="source_record_number")
    return frame, header_line

raw = {}
header_lines = {}
for split, filename in FILES.items():
    raw[split], header_lines[split] = read_source(filename)

features = raw["train"].columns.drop("class").tolist()
assert len(features) == 170
assert raw["test"].columns.tolist() == raw["train"].columns.tolist()
assert {split: len(frame) for split, frame in raw.items()} == {"train": 60000, "test": 16000}
pd.DataFrame({split: {"records": len(frame), "features": len(features),
                      "metadata_lines": header_lines[split]} for split, frame in raw.items()}).T

,records,features,metadata_lines
train,60000,170,20
test,16000,170,20


In [4]:
clean = {}
token_audit_rows = []
for split, frame in raw.items():
    stripped = frame.apply(lambda column: column.str.strip())
    label = stripped["class"].str.lower()
    assert label.isin(["neg", "pos"]).all(), f"Unexpected class label in {split}."
    measurements = stripped[features].replace("na", pd.NA)
    numeric = measurements.apply(pd.to_numeric, errors="coerce").astype("Float64")
    unexpected_tokens = measurements.notna() & numeric.isna()
    assert not unexpected_tokens.any().any(), f"Unrecognised numeric token in {split}."
    clean[split] = pd.concat([label.rename("class"), numeric], axis=1)
    token_audit_rows.append({
        "split": split,
        "whitespace_cells_normalised": int(frame.ne(stripped).sum().sum()),
        "class_labels_normalised": int(stripped["class"].ne(label).sum()),
        "na_tokens_to_missing": int(stripped[features].eq("na").sum().sum()),
        "unexpected_numeric_tokens": int(unexpected_tokens.sum().sum()),
    })
token_audit = pd.DataFrame(token_audit_rows).set_index("split")
token_audit

,whitespace_cells_normalised,class_labels_normalised,na_tokens_to_missing,unexpected_numeric_tokens
split,,,,
train,0,0,850015,0
test,0,0,228680,0


The original files already have consistent labels and parseable measurements. The main
cleaning work is making missingness explicit and establishing a defensible treatment policy,
rather than inventing defects. Source hashes are recorded before any transformation.

In [5]:
summary_rows = []
for split, frame in clean.items():
    x = frame[features]
    summary_rows.append({
        "split": split, "records": len(frame), "features": len(features),
        "APS_cases": int(frame["class"].eq("pos").sum()),
        "APS_sample_share_pct": frame["class"].eq("pos").mean() * 100,
        "missing_measurements": int(x.isna().sum().sum()),
        "missing_measurements_pct": x.isna().sum().sum() / x.size * 100,
        "complete_records": int(x.notna().all(axis=1).sum()),
        "all_missing_records": int(x.isna().all(axis=1).sum()),
    })
dataset_summary = pd.DataFrame(summary_rows).set_index("split")
dataset_summary[["records", "APS_cases", "APS_sample_share_pct",
                 "missing_measurements", "missing_measurements_pct", "complete_records"]]

,records,APS_cases,APS_sample_share_pct,missing_measurements,missing_measurements_pct,complete_records
split,,,,,,
train,60000,1000,1.67,850015,8.33,591
test,16000,375,2.34,228680,8.41,165


## 4. Missing measurements and the cost of row deletion

Missingness is measured across the **170 feature cells per row**, excluding the class label.
The comparison below is descriptive: it does not establish why measurements are absent.
The class labels are used to audit retention; they will not be used to choose imputation values.

In [6]:
class_rows = []
complete_case_rows = []
for split, frame in clean.items():
    row_missing_pct = frame[features].isna().mean(axis=1).mul(100)
    complete = row_missing_pct.eq(0)
    for label in ["neg", "pos"]:
        member = frame["class"].eq(label)
        count = int(member.sum())
        retained = int((member & complete).sum())
        class_rows.append({"split": split, "class": label, "records": count,
                           "mean_missing_pct": row_missing_pct[member].mean(),
                           "median_missing_pct": row_missing_pct[member].median()})
        complete_case_rows.append({"split": split, "class": label,
                                   "source_records": count, "complete_records": retained,
                                   "records_lost": count - retained,
                                   "retained_pct": retained / count * 100})
class_missingness = pd.DataFrame(class_rows)
complete_case_impact = pd.DataFrame(complete_case_rows)
class_missingness

,split,class,records,mean_missing_pct,median_missing_pct
0,train,neg,59000,8.18,4.71
1,train,pos,1000,17.15,12.06
2,test,neg,15625,8.18,4.71
3,test,pos,375,17.86,14.12


In [7]:
complete_case_impact

,split,class,source_records,complete_records,records_lost,retained_pct
0,train,neg,59000,515,58485,0.87
1,train,pos,1000,76,924,7.60
2,test,neg,15625,135,15490,0.86
3,test,pos,375,30,345,8.00


In [8]:
deletion_impact = dataset_summary[["records", "APS_cases", "APS_sample_share_pct"]].copy()
deletion_impact["complete_records"] = dataset_summary["complete_records"]
deletion_impact["all_records_retained_pct"] = (
    deletion_impact["complete_records"] / deletion_impact["records"] * 100
)
deletion_impact["APS_share_after_deletion_pct"] = [
    clean[split].loc[clean[split][features].notna().all(axis=1), "class"].eq("pos").mean() * 100
    for split in deletion_impact.index
]
deletion_impact[["APS_sample_share_pct", "all_records_retained_pct", "APS_share_after_deletion_pct"]].style.format("{:.3f}")

,APS_sample_share_pct,all_records_retained_pct,APS_share_after_deletion_pct
split,,,
train,1.667,0.985,12.860
test,2.344,1.031,18.182


**Decision:** retain incomplete records. Training complete cases have an APS share of
**12.86%**, compared with **1.67%** before deletion, even though most APS cases are lost.
Blanket deletion would therefore discard information and strongly change the class mix.
Neither share is a fleet-wide failure rate.

Missing is also different from zero: a recorded zero is an observed measurement. Filling every
gap with zero would erase that distinction. The conservative cleaned data keeps missing values;
the optional preparation later adds explicit missing indicators before imputation.

In [9]:
band_rows = []
for split, frame in clean.items():
    count_missing = frame[features].isna().sum(axis=1)
    bands = pd.cut(count_missing, bins=[-1, 0, 17, 85, 170],
                   labels=["Complete", ">0-10% missing", ">10-50% missing", ">50% missing"])
    counts = pd.DataFrame({"class": frame["class"], "missing_band": bands}).groupby(
        ["class", "missing_band"], observed=False).size().rename("records").reset_index()
    counts.insert(0, "split", split)
    band_rows.append(counts)
row_missingness_bands = pd.concat(band_rows, ignore_index=True)
row_missingness_bands.pivot(index=["split", "class"], columns="missing_band", values="records")

missing_band  Complete  >0-10% missing  >10-50% missing  >50% missing
split class                                                          
test  neg          135           11849             3532           109
      pos           30             132              208             5
train neg          515           44639            13434           412
      pos           76             369              547             8

## 5. Feature-level quality profile

This profile separates missing cells, actual zeros, negative values, non-finite values and
constant observed values. Without feature units, large numerical values are not automatically
outliers. Negative or non-finite values would stop processing for review; no such values occur
in these pinned source files. No clipping or IQR-based deletion is applied.

In [10]:
profiles = []
for split, frame in clean.items():
    x = frame[features]
    profile = pd.DataFrame({
        "observed_count": x.notna().sum(), "missing_count": x.isna().sum(),
        "missing_pct": x.isna().mean() * 100,
        "zero_count": x.eq(0).sum(), "distinct_observed": x.nunique(dropna=True),
        "min": x.min(), "median": x.median(), "p95": x.quantile(0.95), "max": x.max(),
        "negative_count": x.lt(0).sum(),
        "nonfinite_count": x.isin([float("inf"), float("-inf")]).sum(),
    }).rename_axis("feature").reset_index()
    profile.insert(0, "split", split)
    profiles.append(profile)
feature_profile = pd.concat(profiles, ignore_index=True)
train_profile = feature_profile.query("split == 'train'").set_index("feature")
feature_profile.query("split == 'train'").nlargest(10, "missing_pct")[[
    "feature", "observed_count", "missing_pct", "zero_count", "distinct_observed"]]

,feature,observed_count,missing_pct,zero_count,distinct_observed
78,br_000,10736,82.11,1813,3806
77,bq_000,11278,81.20,1734,4276
76,bp_000,12260,79.57,1612,4968
75,bo_000,13667,77.22,1503,5838
1,ab_000,13671,77.22,10977,29
112,cr_000,13671,77.22,13592,74
74,bn_000,15991,73.35,1422,7096
73,bm_000,20451,65.92,1214,9037
72,bl_000,32723,45.46,1016,11969
71,bk_000,36966,38.39,502,12917


In [11]:
train_missing_rate = clean["train"][features].isna().mean()
sparse_features = train_missing_rate[train_missing_rate.gt(0.50)].index.tolist()
constant_features = train_profile.index[train_profile["distinct_observed"].le(1)].tolist()
feature_flags = pd.DataFrame({
    "train_missing_pct": train_missing_rate * 100,
    "more_than_half_missing": train_missing_rate.gt(0.50),
    "constant_observed_value": train_profile["distinct_observed"].le(1),
}).rename_axis("feature")
feature_flags.loc[feature_flags["more_than_half_missing"] | feature_flags["constant_observed_value"]]

,train_missing_pct,more_than_half_missing,constant_observed_value
feature,,,
ab_000,77.22,True,False
bm_000,65.92,True,False
bn_000,73.35,True,False
bo_000,77.22,True,False
bp_000,79.57,True,False
bq_000,81.20,True,False
br_000,82.11,True,False
cd_000,1.13,False,True
cr_000,77.22,True,False


**Decisions:**

- Keep all 170 original features in the conservative cleaned files. More than 50% missing
  is a transparent review threshold, not a validated cut-off for predictive usefulness.
- `cd_000` has one observed value. Its observed-value column is omitted only from the separate
  preparation example; its missingness indicator is retained if training contains missing values.
- Do not rename anonymised counters into invented business measures or treat unusually large
  values as undocumented sentinels. Feature definitions and sensor rules would be needed.

In [12]:
# Inspect the constant separately: missingness and an observed constant are different facts.
feature_profile.loc[feature_profile["feature"].isin(constant_features), [
    "split", "feature", "missing_count", "missing_pct", "distinct_observed", "min", "max"]]

,split,feature,missing_count,missing_pct,distinct_observed,min,max
89,train,cd_000,676,1.13,1,"1,209,600.00","1,209,600.00"
259,test,cd_000,185,1.16,1,"1,209,600.00","1,209,600.00"


### Histogram completeness

The seven repeated feature prefixes each contain ten bins. The check below verifies the
column sequence and measures records with partially missing bins. It does **not** invent a
total that the bins must sum to: denominators, units and time windows are not supplied.
Partial histograms are retained with their missing bins explicit.

In [13]:
prefixes = pd.Series(features).str.split("_").str[0]
histogram_families = prefixes.value_counts().loc[lambda s: s.gt(1)].index.sort_values().tolist()
assert len(histogram_families) == 7
histogram_rows = []
for family in histogram_families:
    bins = [name for name in features if name.startswith(family + "_")]
    assert bins == [f"{family}_{i:03d}" for i in range(10)]
    for split, frame in clean.items():
        missing_bins = frame[bins].isna().sum(axis=1)
        histogram_rows.append({"split": split, "family": family, "bins": len(bins),
                               "complete_records": int(missing_bins.eq(0).sum()),
                               "partial_records": int(missing_bins.between(1, 9).sum()),
                               "all_bins_missing_records": int(missing_bins.eq(10).sum())})
histogram_completeness = pd.DataFrame(histogram_rows)
histogram_completeness

,split,family,bins,complete_records,partial_records,all_bins_missing_records
0,train,ag,10,59329,0,671
1,test,ag,10,15811,0,189
2,train,ay,10,59329,0,671
3,test,ay,10,15808,0,192
4,train,az,10,59329,0,671
5,test,az,10,15808,0,192
6,train,ba,10,59312,0,688
7,test,ba,10,15807,0,193
8,train,cn,10,59313,0,687
9,test,cn,10,15806,0,194


## 6. Duplicates, conflicting labels and split overlap

Three distinct checks are needed: identical labelled rows within a split, identical feature
rows with conflicting labels, and feature rows appearing in both splits. The overlap comparison
ignores the label and compares all 170 measurements exactly (including missing positions).
It uses an exact pandas merge rather than relying on hash equality.

Even identical rows would require review before removal: the source has no real-world key.
Zero exact overlap also cannot prove that different observations came from different trucks.

In [14]:
duplicate_rows = []
for split, frame in clean.items():
    conflicts = frame.groupby(features, dropna=False, sort=False)["class"].nunique().gt(1)
    duplicate_rows.append({"split": split,
                           "extra_identical_labelled_rows": int(frame.duplicated().sum()),
                           "rows_in_identical_groups": int(frame.duplicated(keep=False).sum()),
                           "feature_groups_with_conflicting_labels": int(conflicts.sum())})
duplicate_audit = pd.DataFrame(duplicate_rows).set_index("split")

train_patterns = clean["train"][features].drop_duplicates().assign(_in_train=True)
overlap = clean["test"][features].merge(train_patterns, on=features, how="left", validate="many_to_one")
overlap_test_records = int(overlap["_in_train"].eq(True).sum())
split_overlap = pd.DataFrame({"check": ["Test records matching a training feature row"],
                              "records": [overlap_test_records]})
del train_patterns, overlap
duplicate_audit

,extra_identical_labelled_rows,rows_in_identical_groups,feature_groups_with_conflicting_labels
split,,,
train,0,0,0
test,0,0,0


In [15]:
split_overlap

,check,records
0,Test records matching a training feature row,0


### Train/test availability comparison

Test data is used here only to audit availability and schema. It is kept separate from
fitting preparation statistics. Similar missingness does not prove matching feature
distributions, stable collection processes or the absence of population shift.

In [16]:
missingness_comparison = pd.DataFrame({
    "train_missing_pct": clean["train"][features].isna().mean() * 100,
    "test_missing_pct": clean["test"][features].isna().mean() * 100,
}).rename_axis("feature")
missingness_comparison["test_minus_train_pp"] = (
    missingness_comparison["test_missing_pct"] - missingness_comparison["train_missing_pct"]
)
missingness_comparison.loc[missingness_comparison["test_minus_train_pp"].abs().nlargest(8).index]

,train_missing_pct,test_missing_pct,test_minus_train_pp
feature,,,
cl_000,15.92,15.37,-0.55
ed_000,15.92,15.37,-0.55
ec_00,17.07,16.53,-0.54
cm_000,16.46,16.11,-0.35
bk_000,38.39,38.09,-0.30
bl_000,45.46,45.16,-0.30
ca_000,7.26,7.54,0.28
ar_000,4.54,4.78,0.24


## 7. Cleaning policy and record lineage

The primary output is a **conservative cleaned dataset**, not a fully observed model matrix.
Missing measurements remain missing. All rows, labels, zeros and observed numerical values
are retained. Separate quality files attach source positions and missingness to every row.

`source_record_number` starts at 1 for the first data record, excluding the metadata and header.
`source_line_number` points to the physical CSV line. Each split has its own record numbering;
`record_id` combines the split and record number for cross-file traceability only.

In [17]:
lineage = {}
for split, frame in clean.items():
    row = pd.DataFrame(index=frame.index)
    row["record_id"] = [f"{split}_{i:06d}" for i in frame.index]
    row["source_file"] = FILES[split]
    row["source_line_number"] = frame.index + header_lines[split] + 1
    row["missing_feature_count"] = frame[features].isna().sum(axis=1)
    row["missing_feature_pct"] = row["missing_feature_count"] / len(features) * 100
    row["identical_labelled_row_flag"] = frame.duplicated(keep=False)
    lineage[split] = row

processing_log = pd.DataFrame([
    ["Source", "Download and verify", "Pin official archive and file hashes; preserve raw bytes."],
    ["Schema", "Validate", "Require identical ordered feature columns, valid labels and numeric tokens."],
    ["Missing values", "Standardise", "Convert documented na tokens to nullable values; preserve observed zero."],
    ["Records", "Retain", "Keep incomplete records and class mix; do not silently deduplicate."],
    ["Features", "Retain", "Keep all original features in conservative clean exports; flag sparse/constant values."],
    ["Extreme values", "Retain", "No clipping without feature units or engineering limits."],
    ["Lineage", "Separate", "Keep source positions and quality flags out of the numerical feature matrix."],
    ["Optional baseline", "Fit on train only", "Apply training medians and training-defined missing indicators to test."],
], columns=["area", "action", "decision"])
processing_log

,area,action,decision
0,Source,Download and verify,Pin official archive and file hashes; preserve raw bytes.
1,Schema,Validate,"Require identical ordered feature columns, valid labels and numeric tokens."
2,Missing values,Standardise,Convert documented na tokens to nullable values; preserve observed zero.
3,Records,Retain,Keep incomplete records and class mix; do not silently deduplicate.
4,Features,Retain,Keep all original features in conservative clean exports; flag sparse/constant values.
5,Extreme values,Retain,No clipping without feature units or engineering limits.
6,Lineage,Separate,Keep source positions and quality flags out of the numerical feature matrix.
7,Optional baseline,Fit on train only,Apply training medians and training-defined missing indicators to test.


## 8. Separate preparation example: training-only median imputation

Some downstream tools require fully observed numerical inputs. This example demonstrates a
reproducible starting point, **not a validated optimal treatment or a trained failure model**.

Training alone determines the constant columns, medians and missing-indicator columns. Test
values and labels are not used to fit any of these choices. Sparse features are retained so
the cleaning exercise does not pre-judge their usefulness. Missing indicators preserve the
distinction between an observed value and a filled gap, including missingness in constant features.

Median filling changes distributions and does not preserve histogram totals. These derived
columns must not be interpreted as measured sensor values or used to calculate physical
histogram shares. Keep the conservative cleaned files for measurement-based analysis.
Model selection, alternative imputation and validation belong to a later project stage.

In [18]:
train_x = clean["train"][features]
train_distinct = train_x.nunique(dropna=True)
value_features = train_distinct[train_distinct.gt(1)].index.tolist()
indicator_features = train_x.columns[train_x.isna().any()].tolist()
train_medians = train_x[value_features].median()
assert train_medians.notna().all(), "A retained feature has no training median."

def prepare_features(frame):
    values = frame[value_features].fillna(train_medians)
    indicators = frame[indicator_features].isna().astype("Int8").add_suffix("__missing")
    return pd.concat([values, indicators], axis=1)

prepared = {split: prepare_features(frame) for split, frame in clean.items()}
feature_policy = pd.DataFrame(index=pd.Index(features, name="feature"))
feature_policy["train_missing_pct"] = train_missing_rate * 100
feature_policy["sparse_review_flag"] = feature_policy.index.isin(sparse_features)
feature_policy["baseline_value_retained"] = feature_policy.index.isin(value_features)
feature_policy["baseline_missing_indicator"] = feature_policy.index.isin(indicator_features)
feature_policy["baseline_training_median"] = train_medians.reindex(features)
pd.DataFrame({split: {"records": len(frame), "value_columns": len(value_features),
                      "missing_indicators": len(indicator_features),
                      "total_feature_columns": frame.shape[1],
                      "remaining_missing_cells": int(frame.isna().sum().sum())}
              for split, frame in prepared.items()}).T

,records,value_columns,missing_indicators,total_feature_columns,remaining_missing_cells
train,60000,169,169,338,0
test,16000,169,169,338,0


## 9. Validation before export

These checks enforce the source contract and the stated transformation policy. In particular,
they verify class retention, unchanged observed values, the separation of labels and lineage
from features, and that every filled test value equals its **training** median. A synthetic
test-only gap checks that an unseen missingness pattern is filled without creating a new
feature column. Such a new pattern should be monitored in deployment.

In [19]:
checks = []

def check(name, condition):
    passed = bool(condition)
    checks.append({"check": name, "passed": passed})
    assert passed, name

for split, frame in clean.items():
    original_numeric = raw[split][features].replace("na", pd.NA).apply(pd.to_numeric).astype("Float64")
    check(f"{split}: row count and source order retained", frame.index.equals(raw[split].index))
    check(f"{split}: all 170 feature names retained", frame.columns.tolist() == ["class", *features])
    check(f"{split}: labels retained", frame["class"].equals(raw[split]["class"]))
    check(f"{split}: observed values and missing positions retained", frame[features].equals(original_numeric))
    check(f"{split}: lineage is one-to-one", lineage[split].index.is_unique and lineage[split].index.equals(frame.index))
    check(f"{split}: no negative or non-finite measurements",
          not frame[features].lt(0).any().any()
          and not frame[features].isin([float("inf"), float("-inf")]).any().any())
    check(f"{split}: prepared features have no missing cells", not prepared[split].isna().any().any())
    check(f"{split}: label and row identifiers excluded from features",
          not set(["class", "record_id", "source_record_number"]).intersection(prepared[split].columns))
    original_values = frame[value_features]
    expected_filled = pd.DataFrame({name: original_values[name].where(
        original_values[name].notna(), train_medians[name]) for name in value_features})
    check(f"{split}: filled values use training medians", prepared[split][value_features].equals(expected_filled))
    expected_indicators = frame[indicator_features].isna().astype("Int8").add_suffix("__missing")
    check(f"{split}: missing indicators describe original values",
          prepared[split][expected_indicators.columns].equals(expected_indicators))

check("Train/test prepared feature order matches", prepared["train"].columns.equals(prepared["test"].columns))
check("Training medians exclude the label", train_medians.index.tolist() == value_features)
check("Published source class counts match", clean["train"]["class"].value_counts().to_dict() == {"neg": 59000, "pos": 1000})

# aa_000 has no training gaps. A future missing value still uses a training median;
# the feature schema does not learn an indicator from the held-out row.
probe = clean["test"].iloc[[0]].copy()
probe.loc[:, "aa_000"] = pd.NA
probe_prepared = prepare_features(probe)
check("Unseen test missingness uses a training median", probe_prepared["aa_000"].iloc[0] == train_medians["aa_000"])
check("Unseen test missingness preserves feature schema", probe_prepared.columns.equals(prepared["train"].columns))
validation_checks = pd.DataFrame(checks)
pd.DataFrame({"checks_passed": [int(validation_checks["passed"].sum())],
              "checks_failed": [int((~validation_checks["passed"]).sum())]})

,checks_passed,checks_failed
0,25,0


## 10. Export the datasets and the audit trail

The repository tracks the executed notebook and compact audit reports. Raw downloads and
large processed CSVs are generated locally and excluded from Git.

| Output | Purpose |
|---|---|
| `data/processed/{split}_clean.csv` | All original labels and 170 features; missing cells written as `NA`. |
| `data/processed/{split}_row_quality.csv` | Source positions, row IDs and quality flags. |
| `data/processed/{split}_baseline_features.csv` | Separate median-filled values and missing indicators; no class label. |
| `data/processed/{split}_labels.csv` | Labels keyed by source record number. |
| `reports/*.csv` | Reconciled quality results, processing decisions and preparation policy. |
| `reports/source_manifest.json` | Official source, exact file hashes, counts and preparation settings. |

All CSVs include a named source-record index where appropriate. Read conservative measurement
files with `na_values="NA"`; use `source_record_number` as a join key **within a split**, never as
a model input. Preparation exports are convenience datasets and do not replace the originals.

In [20]:
output_rows = []
for split, frame in clean.items():
    exports = {
        f"{split}_clean.csv": frame,
        f"{split}_row_quality.csv": lineage[split],
        f"{split}_baseline_features.csv": prepared[split],
        f"{split}_labels.csv": frame[["class"]],
    }
    for filename, table in exports.items():
        path = PROCESSED / filename
        table.to_csv(path, index=True, na_rep="NA")
        output_rows.append({"file": f"data/processed/{filename}", "records": len(table),
                            "data_columns": table.shape[1], "bytes": path.stat().st_size})
export_inventory = pd.DataFrame(output_rows)
export_inventory[["file", "records", "data_columns"]]

,file,records,data_columns
0,data/processed/train_clean.csv,60000,171
1,data/processed/train_row_quality.csv,60000,6
2,data/processed/train_baseline_features.csv,60000,338
3,data/processed/train_labels.csv,60000,1
4,data/processed/test_clean.csv,16000,171
5,data/processed/test_row_quality.csv,16000,6
6,data/processed/test_baseline_features.csv,16000,338
7,data/processed/test_labels.csv,16000,1


In [21]:
# Read the actual exported files back, including their keys and missing-value convention.
for split, frame in clean.items():
    restored = pd.read_csv(PROCESSED / f"{split}_clean.csv", index_col="source_record_number",
                           na_values="NA", dtype={"class": "string", **{name: "Float64" for name in features}})
    check(f"{split}: conservative export round-trip", restored.equals(frame))
    restored_prepared = pd.read_csv(PROCESSED / f"{split}_baseline_features.csv",
                                    index_col="source_record_number", dtype=prepared[split].dtypes.to_dict())
    check(f"{split}: baseline export round-trip", restored_prepared.equals(prepared[split]))
validation_checks = pd.DataFrame(checks)

reports = {
    "dataset_summary.csv": dataset_summary,
    "token_audit.csv": token_audit,
    "feature_quality_profile.csv": feature_profile,
    "class_missingness.csv": class_missingness,
    "complete_case_impact.csv": complete_case_impact,
    "deletion_impact.csv": deletion_impact,
    "row_missingness_bands.csv": row_missingness_bands,
    "histogram_completeness.csv": histogram_completeness,
    "duplicate_audit.csv": duplicate_audit,
    "split_overlap.csv": split_overlap,
    "train_test_missingness_comparison.csv": missingness_comparison,
    "baseline_feature_policy.csv": feature_policy,
    "processing_log.csv": processing_log,
    "validation_checks.csv": validation_checks,
    "export_inventory.csv": export_inventory,
}
for filename, table in reports.items():
    table.to_csv(REPORTS / filename, index=not isinstance(table.index, pd.RangeIndex), na_rep="NA")

manifest = {
    "dataset": "APS Failure at Scania Trucks",
    "source_url": SOURCE_URL, "doi": "https://doi.org/10.24432/C51S51",
    "archive_sha256": ARCHIVE_SHA256, "files": source_manifest.to_dict(orient="records"),
    "pandas_version": pd.__version__, "raw_missing_marker": "na", "export_missing_marker": "NA",
    "record_counts": {split: len(frame) for split, frame in clean.items()},
    "source_feature_count": len(features),
    "preparation": {"fit_split": "train", "value_fill": "training median",
                    "constant_value_features_omitted": constant_features,
                    "missing_indicators": indicator_features,
                    "sparse_features_retained": sparse_features},
}
(REPORTS / "source_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(f"Saved {len(export_inventory)} processed files and {len(reports)} audit CSVs.")
print(f"Validation: {int(validation_checks['passed'].sum())} passed; 0 failed.")

Saved 8 processed files and 15 audit CSVs.
Validation: 29 passed; 0 failed.


## 11. Findings and operational recommendations

| Evidence in this notebook | Recommended action | Boundary of the conclusion |
|---|---|---|
| Complete-case deletion retains 591 of 60,000 training rows and 7.60% of APS cases. | Preserve incomplete records and compare explicit missing-data policies before modelling. | This does not establish which imputation method will predict best. |
| APS cases have 17.15% of features missing on average, versus 8.18% for other failures. | Investigate measurement availability by failure type; retain missingness indicators in a baseline. | Association does not show whether missingness causes, precedes or follows failure. |
| Eight training features exceed 50% missing. | Request collection rules and assess feature usefulness with training-only validation. | Sparse does not mean useless; no automatic deletion is justified here. |
| `cd_000` has a constant observed value. | Exclude its value from the baseline example; retain its availability information. | This is a sample-specific property, not proof that the sensor can never vary. |
| No exact duplicate rows, conflicting exact feature groups or cross-split feature matches were found. | Retain the supplied records and splits; obtain truck IDs and dates before stronger independence claims. | Exact-value checks cannot identify repeated trucks with different readings. |

**What a production handover would need:** truck/event identifiers, observation and failure
timestamps, feature definitions and units, sensor collection rules, and a validated operational
cost framework. With those inputs, the next stage could compare missing-data treatments and
failure detection using truck/time-aware validation and the asymmetric cost of missed APS cases.

**Delivered:** original-source ingestion, a reproducible pandas cleaning process, row-level
lineage, executed quality checks, conservative exports and a clearly separated preparation
example. No fleet prevalence, predictive accuracy or monetary benefit is claimed.